# PPY - Class 06: Decorators & Exception Handling
**Topics:** function decorators, decorator factories, class-based decorators, try/except/finally

## 🧠 Theory

### Basic decorator pattern
```python
import functools

def my_decorator(func):
    @functools.wraps(func)       # preserve metadata
    def wrapper(*args, **kwargs):
        # before
        result = func(*args, **kwargs)
        # after
        return result
    return wrapper

@my_decorator
def greet(name): return f"Hello, {name}"
```

### Decorator factory (with parameters)
```python
def repeat(n):
    def decorator(func):
        @functools.wraps(func)
        def wrapper(*args, **kwargs):
            for _ in range(n):
                result = func(*args, **kwargs)
            return result
        return wrapper
    return decorator

@repeat(3)
def greet(name): print(f"Hello {name}")
```

### Class-based decorator
```python
class Timer:
    def __init__(self, func): self.func = func
    def __call__(self, *args, **kwargs):
        import time
        t0 = time.time()
        r  = self.func(*args, **kwargs)
        print(f"Took {time.time()-t0:.3f}s")
        return r
```

### Exception handling
```python
try:
    risky_code()
except ValueError as e:
    print("Value error:", e)
except (TypeError, KeyError):
    print("Type or Key error")
except Exception as e:
    print("Any error:", e)
else:
    print("No exception occurred")
finally:
    print("Always runs – cleanup here")
```

## ✏️ I – uppercase decorator

In [ ]:
import functools

def uppercase(func):
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        result = func(*args, **kwargs)
        return result.upper() if isinstance(result, str) else result
    return wrapper

@uppercase
def greet(name): return f"hello, {name}!"

print(greet("world"))

## ✏️ II – timer decorator

In [ ]:
import functools, time

def timer(func):
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        t0     = time.time()
        result = func(*args, **kwargs)
        print(f"[timer] {func.__name__} took {time.time()-t0:.4f}s")
        return result
    return wrapper

@timer
def slow_add(a, b):
    time.sleep(0.05)
    return a + b

print("Result:", slow_add(3, 4))

## ✏️ III – repeat(n) decorator factory

In [ ]:
import functools

def repeat(n):
    def decorator(func):
        @functools.wraps(func)
        def wrapper(*args, **kwargs):
            result = None
            for _ in range(n):
                result = func(*args, **kwargs)
            return result
        return wrapper
    return decorator

@repeat(3)
def greet(name): print(f"  Hello {name}!")

greet("Alice")

## ✏️ IV – stacked decorators: @timer + @delay

In [ ]:
import functools, time

def delay(seconds):
    def decorator(func):
        @functools.wraps(func)
        def wrapper(*args, **kwargs):
            time.sleep(seconds)
            return func(*args, **kwargs)
        return wrapper
    return decorator

def timer(func):
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        t0     = time.time()
        result = func(*args, **kwargs)
        print(f"[timer] total time: {time.time()-t0:.3f}s")
        return result
    return wrapper

@timer
@delay(0.05)
def fetch_data():
    return "data"

print("Got:", fetch_data())

## ✏️ V – call_counter

In [ ]:
import functools

def call_counter(func):
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        wrapper.count += 1
        print(f"  [{func.__name__} call #{wrapper.count}]")
        return func(*args, **kwargs)
    wrapper.count = 0
    return wrapper

@call_counter
def add(a, b): return a + b

add(1, 2); add(3, 4); add(5, 6)
print("Total calls:", add.count)

## ✏️ VI – memoize

In [2]:
import functools, time

def memoize(func):
    cache = {}
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        key = (args, tuple(sorted(kwargs.items())))
        if key not in cache:
            cache[key] = func(*args, **kwargs)
        else:
            print(f"  [cache hit] {args}")
        return cache[key]
    return wrapper

@memoize
def slow_add(a, b):
    time.sleep(0.02)
    return a + b

print(slow_add(2, 3))   # computed
print(slow_add(2, 3))   # cached
print(slow_add(3, 4))   # computed

  [cache miss] ((2, 3), ())
5
  [cache miss] ((2, 3), ())
  [cache hit] (2, 3)
5
  [cache miss] ((3, 4), ())
7


## ✏️ VII – type_check decorator

In [ ]:
import functools

def type_check(func):
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        for arg in args:
            if not isinstance(arg, int):
                raise TypeError(f"Argument {arg!r} is not int")
        return func(*args, **kwargs)
    return wrapper

@type_check
def sum_all(*args): return sum(args)

print(sum_all(1, 2, 3))
try:
    sum_all(1, "a", 3)
except TypeError as e:
    print("TypeError:", e)

## ✏️ VIII – safe_execution decorator

In [ ]:
import functools

def safe_execution(func):
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        try:
            return func(*args, **kwargs)
        except Exception as e:
            print(f"[safe_execution] Error in {func.__name__}: {e}")
    return wrapper

@safe_execution
def risky_division(a, b): return a / b

print(risky_division(10, 2))
print(risky_division(10, 0))

## ✏️ IX – auth_required

In [3]:
import functools

def auth_required(func):
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        if kwargs.get("user") != "admin":
            print("  Access denied!")
            return None
        return func(*args, **kwargs)
    return wrapper

@auth_required
def secret_data(user=None): return "TOP SECRET"

print(secret_data(user="admin"))
print(secret_data(user="guest"))

TOP SECRET
  Access denied!
None


## ✏️ X – retry(n) decorator factory

In [ ]:
import functools, random

def retry(n):
    def decorator(func):
        @functools.wraps(func)
        def wrapper(*args, **kwargs):
            for attempt in range(1, n + 1):
                try:
                    return func(*args, **kwargs)
                except Exception as e:
                    print(f"  Attempt {attempt} failed: {e}")
            print("  All attempts failed")
        return wrapper
    return decorator

@retry(3)
def flaky_divide(a, b):
    if random.random() < 0.6: raise ZeroDivisionError("unlucky!")
    return a / b

flaky_divide(10, 2)

## ✏️ Class-based decorators

In [ ]:
import functools, time

class Timer:
    def __init__(self, func):
        functools.update_wrapper(self, func)
        self.func = func
    def __call__(self, *args, **kwargs):
        t0     = time.time()
        result = self.func(*args, **kwargs)
        print(f"[Timer] {self.func.__name__}: {time.time()-t0:.4f}s")
        return result

class Repeat:
    def __init__(self, n): self.n = n
    def __call__(self, func):
        @functools.wraps(func)
        def wrapper(*args, **kwargs):
            for _ in range(self.n): func(*args, **kwargs)
        return wrapper

@Timer
def slow_add(a, b): time.sleep(0.02); return a + b

@Repeat(3)
def greet(name): print(f"  Hello {name}!")

slow_add(2, 3)
greet("Bob")

---
## 📋 Cheatsheet - Class 06

### Decorator pattern
```python
import functools

def my_dec(func):
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        # before
        result = func(*args, **kwargs)
        # after
        return result
    return wrapper
```

### Decorator factory
```python
def repeat(n):
    def decorator(func):
        @functools.wraps(func)
        def wrapper(*args, **kwargs):
            for _ in range(n): func(*args, **kwargs)
        return wrapper
    return decorator
```

### Class-based decorator
```python
class MyDec:
    def __init__(self, func): self.func = func
    def __call__(self, *args, **kwargs):
        return self.func(*args, **kwargs)
```

### Exception handling
```python
try:
    risky()
except ValueError as e: ...      # specific
except (TypeError, KeyError): ... # multiple
except Exception as e: ...        # catch-all
else: ...                         # no exception
finally: ...                      # always runs
```

### Tips
- Always use `*args, **kwargs` in wrappers
- Use `functools.wraps(func)` to keep `__name__`, `__doc__`
- Decorator stacking: bottom decorator applied first
  `@A @B def f()` -> `A(B(f))`
